# FrontEnd(UI)

## Cell1(Model Unzip)

In [ ]:
!unzip -q FineTunned_Ko_LongFormer.zip

## Cell 2(필수 패키지 설치)

In [ ]:
!pip install streamlit groq

In [ ]:
import os

# 1. .streamlit 폴더 강제 생성 (이미 있으면 에러 없이 패스)
os.makedirs(".streamlit", exist_ok=True)

# 2. config.toml 파일에 테마 내용 작성
config_content = """[theme]
base = "light"
primaryColor = "#4F46E5"
backgroundColor = "#FFFFFF"
secondaryBackgroundColor = "#F8FAFC"
textColor = "#0F172A"
font = "sans serif"
"""

# 3. 파일 물리적 저장
with open(".streamlit/config.toml", "w") as f:
    f.write(config_content)

print("✅ 라이트 모드 테마 설정 파일(.streamlit/config.toml)이 완벽하게 생성되었습니다!")

## Cell3(Execution Code)

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import time
import os
import re
from torch.amp import autocast
from groq import Groq
import re

# ---------------------------------------------------
# 페이지 설정 및 트렌디 라이트(Light) 모던 CSS
# ---------------------------------------------------
st.set_page_config(page_title="SLM-LLM Hybrid Architecture Fact Check", page_icon="🧠", layout="wide", initial_sidebar_state="expanded")

st.markdown("""
<style>
/* 1. 폰트 렌더링 최적화 및 여백 조정 */
.block-container { padding-top: 2rem; padding-bottom: 2rem; }
* { -webkit-font-smoothing: antialiased; -moz-osx-font-smoothing: grayscale; }

/* 2. 사이드바 스타일링 (소프트한 경계선과 깔끔한 타이포그래피) */
[data-testid="stSidebar"] {
    border-right: 1px solid #E2E8F0;
}
[data-testid="stSidebar"] h1 {
    font-size: 1.5rem; font-weight: 800; letter-spacing: -0.5px; margin-bottom: 1rem; color: #1E293B;
}
[data-testid="stSidebar"] h3 {
    font-size: 0.9rem; color: #64748B; text-transform: uppercase; letter-spacing: 1px; margin-top: 1.5rem;
}

/* 3. 메인 실행 버튼 - 트렌디한 인디고(Indigo) 입체 섀도우 애니메이션 */
div.stButton > button:first-child {
    background: linear-gradient(135deg, #6366F1 0%, #4F46E5 100%);
    color: white;
    border: none;
    border-radius: 12px;
    height: 3.2rem;
    font-size: 1.05rem;
    font-weight: 600;
    box-shadow: 0 4px 6px -1px rgba(79, 70, 229, 0.2), 0 2px 4px -1px rgba(79, 70, 229, 0.1);
    transition: all 0.2s ease-in-out;
}
div.stButton > button:first-child:hover {
    transform: translateY(-2px);
    box-shadow: 0 10px 15px -3px rgba(79, 70, 229, 0.4);
}

/* 4. 텍스트 입력창 및 알림 박스 모서리 라운딩 처리 */
div.stTextArea > div > div > textarea {
    border-radius: 10px;
    border: 1px solid #CBD5E1;
    background-color: #F8FAFC;
    transition: border-color 0.2s ease;
}
div.stTextArea > div > div > textarea:focus {
    border-color: #6366F1;
    box-shadow: 0 0 0 1px #6366F1;
}
div[data-testid="stAlert"] {
    border-radius: 12px;
    border: 1px solid #E2E8F0;
}
</style>
""", unsafe_allow_html=True)

# ---------------------------------------------------
# [백엔드 연동] Groq API 및 OOV 사전
# ---------------------------------------------------
os.environ["GROQ_API_KEY"] = "Input your API Key"
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

OOV_DICTIONARY = {
    # 1. 언론/기사 특화 관용어 (가장 중요)
    "단독": "단독 보도",
    "속보": "최신 뉴스",
    "종합": "종합 보도",
    "상보": "상세 보도",
    "누리꾼": "인터넷 사용자",
    "네티즌": "인터넷 사용자",
    "입장문": "공식 입장",

    # 2. 경제/금융 뉴스
    "한은": "한국은행",
    "미 연준": "미국 연방준비제도",
    "금감원": "금융감독원",
    "공정위": "공정거래위원회",
    "코스피": "한국 종합주가지수",
    "코스닥": "한국 장외주식시장",
    "종부세": "종합부동산세",
    "금투세": "금융투자소득세",
    "가상자산": "암호화폐",

    # 3. IT/과학 뉴스
    "AI": "인공지능",
    "LLM": "대형언어모델",
    "챗GPT": "ChatGPT",
    "머스크": "일론 머스크",
    "과기부": "과학기술정보통신부",
    "방통위": "방송통신위원회",

    # 4. 사회/노동 뉴스
    "민주노총": "전국민주노동조합총연맹",
    "한국노총": "한국노동조합총연맹",
    "전장연": "전국장애인차별철폐연대",
    "의협": "대한의사협회",
    "전공의": "수련의",
    "학폭": "학교폭력",
    "전세사기": "전세 보증금 사기",

    # 5. 정치/외교 뉴스
    "국힘": "국민의힘",
    "민주": "더불어민주당",
    "대선": "대통령 선거",
    "총선": "국회의원 선거",
    "지선": "지방선거",
    "당정": "여당과 정부",
    "특검": "특별검사",
    "국감": "국정감사",
    "대통령실": "대통령실",
    "합참": "합동참모본부",
    "방사청": "방위사업청",
    "핵잠": "핵잠수함"
}

def normalize_text(text):
    for short, full in OOV_DICTIONARY.items():
        text = text.replace(short, full)
    return text

# 💡 [엔지니어링 최적화] 문맥을 훼손하는 extract_relevant_context 함수는 완전히 삭제했습니다.

def ask_llama_factcheck(premise, hypothesis):
    system_prompt =(
        "[1. 역할 (Role)]\n"
        "당신은 대한민국 최고 수준의 팩트체크 전문 AI 수석 에디터입니다. "
        "객관적인 논리와 문맥 이해력을 바탕으로 가짜뉴스와 낚시성 기사를 정확하게 판별해야 합니다.\n\n"

        "[2. 맥락 (Context)]\n"
        "당신에게는 1차 AI 필터가 판독을 보류한, 다소 교묘하고 판단하기 까다로운 '기사 본문(Evidence)'과 '기사 제목(Claim)'이 주어집니다. "
        "주의: 기사 제목에 사용된 따옴표(' ', \" \")나 축약어, 비유적 표현은 언론의 정상적인 편집 관행일 수 있습니다. "
        "단순한 '어휘의 불일치'가 아니라, 실제 팩트가 충돌하는 '논리적 모순'이나 '과장(낚시)'이 있는지를 파악해야 합니다.\n\n"

        "[3. 수행 작업 (Task)]\n"
        "주어진 '기사 본문'과 '기사 제목'을 꼼꼼히 대조하여 다음 세 가지 중 하나로 판별하십시오.\n"
        "1. 진짜 뉴스: 제목이 본문의 내용을 논리적으로 정확히 반영함.\n"
        "2. 가짜뉴스: 제목이 본문의 내용과 정면으로 충돌하거나, 없는 사실을 낚시성으로 과장함.\n"
        "3. 판단 불가: 주어진 본문의 내용만으로는 제목의 진위 여부를 도저히 판단할 수 없거나 아예 무관한 내용임.\n\n"

        "[4. 출력 형식 및 제약 사항 (Format/Constraints)]\n"
        "- 절대 억지로 추론하여 정답을 끼워 맞추지 마십시오. 본문에 근거가 없다면 반드시 '판단 불가'로 판정해야 합니다.\n"
        "- 높은 정답률을 위해 판정 전에 반드시 논리적 분석(사고 과정)을 먼저 거치십시오.\n"
        "- 답변은 반드시 아래의 지정된 포맷을 엄격히 준수하여 출력하십시오.\n\n"

        "[판단 예시 1]\n"
        "본문: 박명수가 라디오에서 선거에 대해 이야기했다.\n"
        "제목: 박명수 '사람 잘못 뽑으면 큰일나'\n"
        "[사고 과정]: 본문에 구체적인 인용구는 없으나, 선거 관련 이야기라는 문맥상 제목의 인용구는 언론의 정상적인 축약 및 인용 관행으로 볼 수 있어 모순되지 않음.\n"
        "[판정]: 진짜 뉴스\n\n"

        "[판단 예시 2]\n"
        "본문: 경찰 조사 결과, 해당 유명인의 횡령 의혹은 사실무근으로 밝혀졌으며 무혐의 처분을 받았다.\n"
        "제목: [단독] 유명인 A씨 수백억 횡령 혐의 인정… 결국 구속되나\n"
        "[사고 과정]: 본문에서는 명확히 '사실무근' 및 '무혐의'라고 밝혔으나, 제목은 혐의를 인정하고 구속될 것처럼 정반대의 내용을 서술하여 논리적 모순이 발생함.\n"
        "[판정]: 가짜뉴스\n\n"

        "[판단 예시 3]\n"
        "본문: 서울에 비가 내린다.\n"
        "제목: 전국 부동산 가격 폭락\n"
        "[사고 과정]: 주어진 본문의 '비'라는 기상 정보와 제목의 '부동산 가격 폭락' 사이에는 어떠한 논리적 연관성도 없음.\n"
        "[판정]: 판단 불가\n\n"

        "[최종 출력 포맷]\n"
        "[사고 과정]: (한 줄로 간결하고 명확한 논리적 근거 제시)\n"
        "[판정]: ('진짜 뉴스', '가짜뉴스', '판단 불가' 중 택 1)"
    )

    try:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": f"기사 본문: {premise}\n\n기사 제목: {hypothesis}\n\n⚠️ 지시사항: 다른 말은 일절 하지 말고, 오직 '[사고 과정]:' 과 '[판정]:' 두 가지만 지정된 포맷에 맞춰 출력하십시오."
                }
            ],
            model="llama-3.3-70b-versatile",
            temperature=0.0,
            max_tokens=256,  # 토큰 수를 줄여 쓸데없는 말을 방지합니다.
        )

        result = chat_completion.choices[0].message.content.strip()

        # [매우 간단한 텍스트 후처리 방어 로직]
        # 모델이 포맷을 어기고 '[판정]' 이라는 단어를 빼먹었을 때만 실행됩니다.
        if "[판정]" not in result:
            if "가짜" in result or "모순" in result or "충돌" in result:
                result += "\n\n[판정]: 가짜뉴스"
            elif "판단 불가" in result or "무관" in result:
                result += "\n\n[판정]: 판단 불가"
            else:
                result += "\n\n[판정]: 진짜 뉴스"

        return result

    except Exception as e:
        return f"🚨 LLM API 호출 에러: {e}"

# ---------------------------------------------------
# 모델 로드 (FineTunned Klue-RoBERTa)
# ---------------------------------------------------
@st.cache_resource
def load_nli_model():
    model_path = "."
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return tokenizer, model, device

# ---------------------------------------------------
# 사이드바 (모던 라이트 UI 및 커스텀 카드 적용)
# ---------------------------------------------------
with st.sidebar:
    st.title("News FactCheck AI")
    st.markdown("---")

    st.subheader("Architecture")
    # 💡 [수정] 1차 필터 모델명을 Ko-BigBird로 변경
    st.info("**SLM Filter**\n\nKo-BigBird (1차 필터)\n\n**LLM Routing**\n\nLlama 3.3 (교차 검증)\n\n**Tech Stack**\n\nPyTorch · Streamlit · Groq")

    st.markdown("---")

    st.subheader("Configuration")
    # 💡 [핵심 수정] value 기본값을 0.95 -> 0.80으로 변경하고 안내 문구 수정
    threshold = st.slider(
        "SLM Confidence Threshold",
        min_value=0.50, max_value=1.00, value=0.80, step=0.01,
        help="권장 임계값은 0.80 ~ 0.85입니다. 이보다 낮으면 오판 확률이 커지고, 1.0에 가까우면 단순 필터링도 LLM이 처리하게 되어 응답이 느려집니다."
    )
    st.caption("💡 **엔지니어 권장값: 0.80**\n\nSLM의 빠른 처리 속도와 LLM의 정밀한 교차 검증을 완벽하게 조율하는 세팅입니다.")

    st.markdown("---")

    st.subheader("Labels")
    st.markdown("""
    <div style='padding: 12px; border-radius: 10px; background-color: #EEF2FF; border: 1px solid #C7D2FE; margin-bottom: 10px;'>
        <div style='font-size: 1rem; font-weight: 800; color: #4F46E5; margin-bottom: 4px;'>✅ Entailment (참)</div>
        <div style='font-size: 0.85rem; color: #6366F1;'>검증 주장이 뉴스 기사 본문과 논리적으로 완벽히 일치</div>
    </div>

    <div style='padding: 12px; border-radius: 10px; background-color: #FFF1F2; border: 1px solid #FECDD3;'>
        <div style='font-size: 1rem; font-weight: 800; color: #E11D48; margin-bottom: 4px;'>❌ Contradiction (거짓)</div>
        <div style='font-size: 0.85rem; color: #F43F5E;'>검증 주장이 뉴스 기사 본문의 팩트와 정면으로 충돌</div>
    </div>
    """, unsafe_allow_html=True)


# ---------------------------------------------------
# 메인 UI
# ---------------------------------------------------
st.markdown("<h1 style='text-align: center; color: #0F172A; margin-bottom: 0.5rem; font-size: 2.5rem; font-weight: 900;'>Fact News Checking 💡</h1>", unsafe_allow_html=True)
st.markdown("<p style='text-align: center; color: #64748B; font-size: 1.1rem; font-weight: 600; margin-bottom: 2rem;'>SLM-LLM Cascade 아키텍처 기반 실시간 교차 검증 엔진</p>", unsafe_allow_html=True)
st.divider()
st.divider()

col1, col2 = st.columns(2)
with col1: claim_input = st.text_area("💬 뉴스 기사 제목 (Claim)", height=200)
with col2: evidence_input = st.text_area("📰 뉴스 기사 본문 전문 (Evidence)", height=200) # 💡 [수정] 텍스트 가이드 변경
st.divider()

if st.button("🚀 실시간 문맥 검증 시작", use_container_width=True):
    if claim_input.strip() == "" or evidence_input.strip() == "":
        st.warning("⚠️ Claim과 Evidence를 모두 입력해주세요.")
    else:
        tokenizer, model, device = load_nli_model()
        loading_message = st.empty()

        loading_message.info("🧹 텍스트 정규화 및 추론 연산 중...")

        # 💡 [핵심 수정] 문장 절삭 함수(extract_relevant_context) 제거. 기사 전문을 통째로 사용!
        refined_premise = normalize_text(evidence_input)
        refined_hypothesis = normalize_text(claim_input)

        start_time = time.time()

        # 💡 [핵심 수정] max_length를 512에서 1024로 확장
        inputs = tokenizer(refined_premise, refined_hypothesis, return_tensors="pt", truncation=True, padding=True, max_length=1024).to(device)
        inputs['input_ids'] = torch.clamp(inputs['input_ids'], min=0, max=model.config.vocab_size - 1)

        with torch.no_grad():
            with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                # BigBird Tokenizer가 알아서 input_ids, attention_mask, token_type_ids를 생성하므로 **inputs로 전달
                outputs = model(**inputs)
                probs = F.softmax(outputs.logits, dim=-1).squeeze(0).detach().cpu().numpy()

        loading_message.empty()

        p_entail, p_contra = probs[0], probs[1]

        pred_class = probs.argmax()
        slm_confidence = probs[pred_class] # 💡 [수정] 변수명 일원화

        st.success(f"🎯 1차 연산 완료! ({time.time() - start_time:.2f}초)")
        st.divider()
        st.subheader("🧾 AI 최종 결론")

        if slm_confidence >= threshold:
            if pred_class == 0:
                st.success(f"✅ 신뢰 가능한 정보 확정 (확신도: {slm_confidence * 100:.1f}%)")
                final_result_text = "참 (Entailment) - 근거 자료와 일치합니다."
            else:
                st.error(f"🚨 가짜뉴스 판단 확정 (확신도: {slm_confidence * 100:.1f}%)")
                final_result_text = "모순 (Contradiction) - 근거 자료와 정면으로 충돌합니다."
            is_routed = False
        else:
            st.warning(f"⚠️ [판독 보류] 신뢰도({slm_confidence * 100:.1f}%)가 임계값 미만입니다.")
            with st.spinner("🧠 Llama 3.3 MaaS로 라우팅 중..."):
                llama_result = ask_llama_factcheck(refined_premise, refined_hypothesis)
            st.info(f"🤖 **Llama 3.3 결과**\n\n{llama_result}")
            final_result_text = "LLM 교차 검증 대행 완료"
            is_routed = True

        st.divider()
        tab1, tab2 = st.tabs(["📊 분석 리포트", "💡 시스템 진단"])

        with tab1:
            st.write("**SLM 예측 확률**")
            st.progress(int(p_entail * 100), text=f"✅ 참 ({p_entail * 100:.1f}%)")
            st.progress(int(p_contra * 100), text=f"❌ 거짓 ({p_contra * 100:.1f}%)")

        with tab2:
            st.write(f"**라우팅 여부:** {'🚨 TRUE (Llama 호출됨)' if is_routed else '✅ FALSE (SLM 단독 처리)'}")
            st.write(f"**Ko-BigBird 최대 확신도:** {slm_confidence * 100:.1f}%") # 💡 [수정] 모델명 변경
            st.write(f"**검증에 사용된 본문 전문:** {refined_premise}")

## Cell 4 Cloudflare 터널 가동

In [ ]:
import subprocess
import time
import re  # 정규표현식 모듈 추가

# 1. Cloudflare 다운로드 및 권한 부여
print("📥 [1/4] Cloudflare 터널링 인프라 다운로드 중...")
!wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# 2. 기존 프로세스 청소
print("🧹 [2/4] 포트 충돌 방지를 위한 기존 프로세스 정화 중...")
!pkill -f streamlit
!pkill -f cloudflared
time.sleep(2)

# 3. Streamlit 웹 서버 백그라운드 구동
print("🚀 [3/4] 로컬 Streamlit 웹 서버 구동 시작...")
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])
time.sleep(4)

# 4. Cloudflare 터널 실행 (tunnel.log 파일에 기록)
print("🌐 [4/4] 글로벌 터널 생성 및 URL 추적 중 (약 7초 대기)...")
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8501 > tunnel.log 2>&1 &

time.sleep(7) # 로그 파일에 주소가 찍힐 때까지 넉넉히 대기

# 5. 정규표현식(RegEx)으로 오직 URL만 완벽하게 추출
url_found = False
with open("tunnel.log", "r") as f:
    log_content = f.read()
    # https:// 로 시작하고 .trycloudflare.com 으로 끝나는 패턴만 정확히 찾음
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)

    if match:
        url = match.group(0)
        print("\n" + "="*70)
        print("🎉 [배포 성공] 아래 링크를 클릭하여 팩트체커에 접속하세요! 🎉")
        print(f"👉 {url}")
        print("="*70)
        url_found = True

if not url_found:
    print("⏳ 아직 터널 주소가 발급되지 않았습니다. (Cloudflare 서버 지연)")
    print("10초 뒤에 코랩 빈 셀을 하나 만들고 [ !cat tunnel.log ] 를 실행하시면 주소를 찾으실 수 있습니다.")